# Ana deney kuyruğu

19 koşunun (2×2 tasarım + ablasyon) hepsini sırayla eğitir ve her birini beş test kümesinde değerlendirir. **Tümünü çalıştır** demek yeterlidir; biten koşular atlanır, yarım kalan koşu kaldığı epoch'tan devam eder. Oturum kapanırsa notebook yeniden açılıp tekrar çalıştırılır.

**Sıra (dengeli):** önce her yapılandırmanın seed 1'i, sonra seed 2'si, sonra seed 3'ü; en son Res2Net hücrelerinin seed 4 ve 5'i. Böylece kuyruk herhangi bir noktada yarıda kalsa bile tasarım dengeli kalır ve bütçe yetmezse önce Res2Net'in ek seed'leri düşer (öneride yazılan kural).

**Sabit ayarlar:** GPU tipi ve cuDNN kararı pilotta `config.json`'a yazıldı (L4, `--cudnn 1`). Bu oturumun GPU'su farklıysa hücre durur.

**Not (test kümesi):** Her koşu bittiğinde test kümelerinde değerlendirilir; eğitim tarifi pilotta, kendi koşularımızın hiçbir test sonucu görülmeden dondurulduğu için bu değerlendirmeler hiçbir kararı etkilemez.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/PraNet_proje'
GITHUB_USER, BRANCH = 'esmapasaoglu', 'proje'

import os, sys, json, shutil, subprocess, platform, time, glob, zipfile
from google.colab import drive
drive.mount('/content/drive')
REPO_DIR = '/content/PraNet-V2'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH,
                    f'https://github.com/{GITHUB_USER}/PraNet-V2.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(f'{REPO_DIR}/binary_seg')
sys.path.insert(0, os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

!pip -q install timm thop tabulate gdown
import torch
GPU = torch.cuda.get_device_name(0)
cfg = json.load(open(f'{DRIVE_ROOT}/config.json'))
CUDNN = cfg['cudnn']
assert GPU == cfg['gpu'], f"GPU uyuşmuyor: bu oturum {GPU}, karar {cfg['gpu']}. Çalışma zamanını {cfg['gpu']} yapın."
print('GPU', GPU, '| cudnn', CUDNN)

# Omurgalar ve veri: Drive'dan yerel diske
os.makedirs('models', exist_ok=True)
for name in ['pvt_v2_b2.pth', 'res2net50_v1b_26w_4s-3cf99910.pth']:
    if not os.path.exists(f'models/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/models/{name}', f'models/{name}')
LOCAL = '/content/data'
if not os.path.isdir(f'{LOCAL}/TrainDataset'):
    for z in sorted(glob.glob(f'{DRIVE_ROOT}/data_raw/*.zip')):
        zipfile.ZipFile(z).extractall(LOCAL)
TRAIN, TEST = f'{LOCAL}/TrainDataset', f'{LOCAL}/TestDataset'
print('Veri hazır:', len(os.listdir(f'{TRAIN}/image')), 'eğitim görüntüsü')

def run(args, log_file=None, quiet=False):
    cmd = [sys.executable, '-W', 'ignore', '-u'] + args
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    f = open(log_file, 'a') if log_file else None
    for line in p.stdout:
        lines.append(line)
        if f:
            f.write(line); f.flush()
        if not quiet and ('Epoch' in line and 'done' in line or 'mDice' in line or 'Resumed' in line):
            print('   ' + line, end='')
    p.wait()
    if f:
        f.close()
    if p.returncode != 0:
        print(''.join(lines[-30:]))
        raise RuntimeError('Komut hata verdi: ' + ' '.join(args))

In [ ]:
# (yapılandırma adı, model, bg_weight)
CONFIGS = [('PraNet-V1', 'PraNet-V1', 0.8), ('PraNet-V2', 'PraNet-V2', 0.8),
           ('PVT-PraNet-V1', 'PVT-PraNet-V1', 0.8), ('PVT-PraNet-V2', 'PVT-PraNet-V2', 0.8),
           ('PVT-PraNet-V2-abl', 'PVT-PraNet-V2', 0.0)]
QUEUE = [(c, s) for s in (1, 2, 3) for c in CONFIGS]
QUEUE += [(c, s) for s in (4, 5) for c in CONFIGS if c[1] in ('PraNet-V1', 'PraNet-V2')]

def status():
    done = [f'{c[0]}_s{s}' for c, s in QUEUE
            if os.path.exists(f'{DRIVE_ROOT}/results/runs/{c[0]}_s{s}/summary.json')]
    print(f'Tamamlanan: {len(done)}/{len(QUEUE)}')
    return done
status()

for (name, mt, bg), seed in QUEUE:
    tag = f'{name}_s{seed}'
    RUN = f'{DRIVE_ROOT}/runs/{tag}'
    EVAL = f'{DRIVE_ROOT}/results/runs/{tag}'
    if os.path.exists(f'{EVAL}/summary.json'):
        continue
    t0 = time.time()
    print(f"\n===== {tag}  (başlangıç {time.strftime('%H:%M')}) =====")
    if not os.path.exists(f'{RUN}/last.pth'):
        run(['train_seed.py', '--model_type', mt, '--seed', str(seed), '--bg_weight', str(bg),
             '--out_dir', RUN, '--train_path', TRAIN, '--save_every', '0', '--cudnn', str(CUDNN)],
            log_file=f'{DRIVE_ROOT}/logs/{tag}.txt')
    run(['eval_per_image.py', '--model_type', mt, '--ckpt', f'{RUN}/last.pth', '--out_dir', EVAL,
         '--data_root', TEST], log_file=f'{DRIVE_ROOT}/logs/{tag}_eval.txt')
    print(f'   bitti: {(time.time() - t0) / 60:.0f} dk')
status()
print('KUYRUK TAMAMLANDI')